# Trích xuất Tư thế 3D từ Video bằng GVHMR

Notebook này hướng dẫn cách sử dụng GVHMR (vm_video2robot) để trích xuất chuyển động 3D
của người từ video. Đây là bước đầu tiên trong pipeline data.

## Yêu cầu
- Đã cài đặt môi trường `video2robot` (xem doc 02)
- Đã tải checkpoint GVHMR và các model phụ trợ
- Có file video đầu vào (xem doc 04 về cách quay video)

## 1. Kích hoạt Môi trường

Trước khi chạy notebook này, đảm bảo bạn đã kích hoạt đúng môi trường conda.
Chạy trong terminal:

```bash
conda activate video2robot
```

Sau đó mở lại Jupyter notebook từ môi trường đó.

In [ ]:
# Thiết lập đường dẫn
import os
import sys

# Đường dẫn repo vm_video2robot
VIDEO2ROBOT_DIR = "/home/nguyenld12/Documents/Humanoid_Tracking_Task/vm_video2robot"
os.chdir(VIDEO2ROBOT_DIR)
sys.path.insert(0, VIDEO2ROBOT_DIR)

print(f"Thư mục làm việc: {os.getcwd()}")

In [ ]:
# Kiểm tra GPU và các thư viện
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

## 2. Chuẩn bị Video Đầu vào

Đặt đường dẫn video bạn muốn xử lý. Video nên:
- Có 1 người duy nhất
- Toàn bộ cơ thể trong khung hình
- Camera tĩnh (hoặc ít di chuyển)

In [ ]:
# === SỬA ĐƯỜNG DẪN VIDEO TẠI ĐÂY ===
VIDEO_PATH = "docs/example_video/tennis.mp4"  # Thay bằng video của bạn

# Camera có cố định không? True = cố định, False = di chuyển
STATIC_CAM = True

# Kiểm tra file tồn tại
assert os.path.exists(VIDEO_PATH), f"Không tìm thấy video: {VIDEO_PATH}"
print(f"Video: {VIDEO_PATH}")
print(f"Camera tĩnh: {STATIC_CAM}")

In [ ]:
# Xem thông tin video
from hmr4d.utils.video_io_utils import get_video_lwh
from pathlib import Path

video_path = Path(VIDEO_PATH)
length, width, height = get_video_lwh(video_path)
print(f"Số khung hình: {length}")
print(f"Kích thước: {width} x {height}")
print(f"Thời lượng ước tính: {length / 30:.1f} giây (giả sử 30fps)")

## 3. Chạy GVHMR

Bước này sẽ:
1. Phát hiện người trong video (YOLO)
2. Ước lượng tư thế 2D (ViTPose)
3. Nâng cấp lên 3D (GVHMR)
4. Ước lượng chuyển động camera (nếu cần)

Thời gian xử lý: khoảng 1-5 phút tùy độ dài video và GPU.

In [ ]:
# Cách 1: Chạy trực tiếp bằng command line (khuyến nghị)
# Chạy cell này và đợi kết quả

static_flag = "-s" if STATIC_CAM else ""
cmd = f"python tools/demo/demo.py --video={VIDEO_PATH} {static_flag}"
print(f"Đang chạy: {cmd}")
print("Vui lòng đợi...")

os.system(cmd)

## 4. Kiểm tra Kết quả

GVHMR lưu kết quả trong thư mục `outputs/demo/`. 
Kiểm tra xem file đầu ra đã tạo thành công chưa.

In [ ]:
# Liệt kê kết quả
output_dir = f"outputs/demo/{video_path.stem}"
if os.path.exists(output_dir):
    print(f"Thư mục kết quả: {output_dir}")
    print("\nCác file:")
    for f in sorted(os.listdir(output_dir)):
        size_mb = os.path.getsize(os.path.join(output_dir, f)) / 1024 / 1024
        print(f"  {f} ({size_mb:.2f} MB)")
else:
    print(f"[LỖI] Không tìm thấy thư mục: {output_dir}")
    print("GVHMR có thể đã gặp lỗi. Kiểm tra output ở cell trước.")

## 5. Trực quan hóa Kết quả

GVHMR tạo video so sánh giữa video gốc và kết quả 3D.
Bạn có thể xem trong thư mục output hoặc hiển thị ngay trong notebook.

In [ ]:
# Hiển thị video kết quả (nếu có)
from IPython.display import Video, display

result_videos = [f for f in os.listdir(output_dir) if f.endswith('.mp4')]
if result_videos:
    result_video = os.path.join(output_dir, result_videos[0])
    print(f"Video kết quả: {result_video}")
    display(Video(result_video, embed=True, width=640))
else:
    print("Không tìm thấy video kết quả.")

## 6. Xử lý Nhiều Video

Nếu bạn có nhiều video cần xử lý, dùng script xử lý cả thư mục.
Đặt tất cả video vào một thư mục, rồi chạy:

In [ ]:
# Xử lý cả thư mục
# Bỏ comment và sửa đường dẫn để chạy

# INPUT_FOLDER = "inputs/demo/folder_in"   # Thư mục chứa video
# OUTPUT_FOLDER = "outputs/demo/folder_out" # Thư mục kết quả

# cmd = f"python tools/demo/demo_folder.py -f {INPUT_FOLDER} -d {OUTPUT_FOLDER} -s"
# os.system(cmd)

## 7. Xử lý Lỗi Thường gặp

| Lỗi | Nguyên nhân | Giải pháp |
|-----|-------------|----------|
| CUDA out of memory | Video quá dài hoặc GPU thiếu VRAM | Cắt video ngắn hơn |
| No person detected | YOLO không phát hiện người | Kiểm tra ánh sáng, người đủ rõ |
| Tư thế sai lệch | Video mờ hoặc bị che khuất | Quay lại với điều kiện tốt hơn |
| ModuleNotFoundError | Chưa cài đủ thư viện | Kiểm tra đã kích hoạt đúng conda env |

## Bước tiếp theo

Sau khi có kết quả SMPL, chuyển sang bước **Retargeting** để chuyển đổi thành motion cho robot M2v6.
Xem [doc 05 - Hướng dẫn Retargeting](../docs/05-huong-dan-retargeting.md).